# 문맥 압축 검색 — Runnable과 구조화 출력

검색된 문서를 그대로 LLM에 넣지 않고 질의와 관련된 문서만 고르거나, 관련
구절만 추출하거나, 임베딩으로 저비용 필터링합니다. 예전의
`ContextualCompressionRetriever`/`LLMChainExtractor` 대신 현재의 Runnable과
`with_structured_output()`을 조합해 각 단계를 투명하게 구현합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain==1.4.2" "langchain-core==1.6.3" \
#   "langchain-openai==1.6.2" "langchain-chroma==1.1.0" \
#   "langchain-text-splitters==1.1.2" python-dotenv numpy


In [ ]:
import getpass
import os
from pathlib import Path
from uuid import uuid4

import numpy as np
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from pydantic import BaseModel, Field

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.4-mini")
llm = init_chat_model(f"openai:{CHAT_MODEL}", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


In [ ]:
def pretty_print_docs(docs: list[Document]) -> None:
    for index, doc in enumerate(docs, start=1):
        print(f"문서 {index}:\n{doc.page_content}\n{'-' * 100}")


data_path = Path("data/appendix-keywords.txt")
if not data_path.exists():
    raise FileNotFoundError(f"실습 파일이 없습니다: {data_path.resolve()}")

raw_docs = [
    Document(
        page_content=data_path.read_text(encoding="utf-8"),
        metadata={"source": str(data_path)},
    )
]
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0)
chunks = splitter.split_documents(raw_docs)
vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    collection_name=f"compression-demo-{uuid4().hex}",
    collection_configuration=CHROMA_CONFIGURATION,
)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

query = "Semantic Search에 대해서 알려주세요."
base_docs = base_retriever.invoke(query)
pretty_print_docs(base_docs)


## 1. LLM 문서 필터

각 문서를 `relevant: bool`로 판정합니다. 자유 형식 문자열을 파싱하지 않고
Pydantic 스키마로 검증된 결과를 받습니다.


In [ ]:
class RelevanceGrade(BaseModel):
    relevant: bool = Field(description="질문에 답하는 데 문서가 유용하면 true")


grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "질문과 문서의 관련성을 엄격하게 판정하세요."),
        ("user", "질문: {question}\n\n문서:\n{document}"),
    ]
)
grade_chain = grade_prompt | llm.with_structured_output(RelevanceGrade)


def llm_filter(question: str, docs: list[Document]) -> list[Document]:
    inputs = [
        {"question": question, "document": doc.page_content} for doc in docs
    ]
    grades = grade_chain.batch(inputs, config={"max_concurrency": 5})
    return [doc for doc, grade in zip(docs, grades) if grade.relevant]


filtered_docs = llm_filter(query, base_docs)
pretty_print_docs(filtered_docs)


## 2. LLM 관련 구절 추출

문서 전체를 유지하는 필터와 달리, 답변에 필요한 구절만 새 `Document`로 만듭니다.
원본 메타데이터는 보존합니다.


In [ ]:
class RelevantExcerpt(BaseModel):
    relevant: bool = Field(description="관련 구절이 있으면 true")
    excerpt: str = Field(
        default="",
        description="원문에서 뽑은 관련 구절. 관련 내용이 없으면 빈 문자열",
    )


extract_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "질문에 답하는 데 필요한 문장만 원문의 의미를 바꾸지 않고 추출하세요.",
        ),
        ("user", "질문: {question}\n\n문서:\n{document}"),
    ]
)
extract_chain = extract_prompt | llm.with_structured_output(RelevantExcerpt)


def llm_extract(question: str, docs: list[Document]) -> list[Document]:
    inputs = [
        {"question": question, "document": doc.page_content} for doc in docs
    ]
    outputs = extract_chain.batch(inputs, config={"max_concurrency": 5})
    compressed: list[Document] = []
    for doc, output in zip(docs, outputs):
        if output.relevant and output.excerpt.strip():
            compressed.append(
                Document(
                    page_content=output.excerpt.strip(),
                    metadata={**doc.metadata, "compressed": True},
                )
            )
    return compressed


extracted_docs = llm_extract(query, base_docs)
pretty_print_docs(extracted_docs)


## 3. 임베딩 필터와 중복 제거

LLM 호출 전에 싼 단계로 후보를 줄입니다. 임계값은 데이터와 임베딩 모델마다
다르므로 검증 셋으로 조정해야 합니다.


In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denominator) if denominator else 0.0


def embedding_filter(
    question: str,
    docs: list[Document],
    threshold: float = 0.55,
) -> list[Document]:
    if not docs:
        return []
    query_vector = np.asarray(embeddings.embed_query(question))
    doc_vectors = embeddings.embed_documents([doc.page_content for doc in docs])
    kept: list[Document] = []
    for doc, vector in zip(docs, doc_vectors):
        score = cosine_similarity(query_vector, np.asarray(vector))
        if score >= threshold:
            kept.append(
                Document(
                    page_content=doc.page_content,
                    metadata={**doc.metadata, "query_similarity": score},
                )
            )
    return kept


def remove_redundant(
    docs: list[Document],
    similarity_threshold: float = 0.95,
) -> list[Document]:
    if not docs:
        return []
    vectors = [np.asarray(v) for v in embeddings.embed_documents(
        [doc.page_content for doc in docs]
    )]
    selected_docs: list[Document] = []
    selected_vectors: list[np.ndarray] = []
    for doc, vector in zip(docs, vectors):
        is_duplicate = any(
            cosine_similarity(vector, previous) >= similarity_threshold
            for previous in selected_vectors
        )
        if not is_duplicate:
            selected_docs.append(doc)
            selected_vectors.append(vector)
    return selected_docs


embedding_filtered_docs = embedding_filter(query, base_docs)
pretty_print_docs(embedding_filtered_docs)


## 4. 압축 파이프라인을 Runnable로 합성

검색 → 더 작은 분할 → 중복 제거 → 임베딩 필터 → LLM 구절 추출 순서입니다.
각 함수는 단독 평가가 가능하고 전체 파이프라인도 `invoke()`할 수 있습니다.


In [ ]:
fine_splitter = CharacterTextSplitter(chunk_size=180, chunk_overlap=20)


def compression_pipeline(question: str) -> list[Document]:
    retrieved = base_retriever.invoke(question)
    fine_chunks = fine_splitter.split_documents(retrieved)
    deduplicated = remove_redundant(fine_chunks)
    relevant = embedding_filter(question, deduplicated, threshold=0.55)
    return llm_extract(question, relevant)


compression_retriever = RunnableLambda(compression_pipeline).with_config(
    {"run_name": "contextual_compression"}
)
compressed_docs = compression_retriever.invoke(query)
pretty_print_docs(compressed_docs)
